# Train the voxel VAE + visualize the latent space

In [ ]:
!pip install -q git+https://github.com/max3925vats/ProteinPatchAnalysis.git

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import pickle, torch, matplotlib.pyplot as plt
from protein_patch.spec import PatchSpec
from protein_patch.config import TrainConfig
from protein_patch.model.train import train
from protein_patch.model.vae import ConvVAE3D
from protein_patch.model.dataset import PatchDataset
from protein_patch.analysis import embed_patches, pca_2d

DATA = Path('/content/drive/MyDrive/protein_patches')
spec = PatchSpec()

In [ ]:
for kl in [0.1, 1.0, 10.0]:
    cfg = TrainConfig(epochs=30, batch_size=32, kl_weight=kl, latent_dim=4,
                      checkpoint_path=f'/content/best_kl{kl}.pt')
    hist = train(str(DATA/'train'), str(DATA/'val'), spec, cfg)
    plt.plot(hist['val_loss'], label=f'kl={kl}')
plt.legend(); plt.xlabel('epoch'); plt.ylabel('val loss'); plt.show()

In [ ]:
model = ConvVAE3D(spec, latent_dim=4)
ckpt = torch.load('/content/best_kl1.0.pt', weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])

ds = PatchDataset(DATA/'val', spec)
emb = embed_patches(model, ds)
xy = pca_2d(emb)
attrs = [pickle.load(open(p, 'rb')).attrs for p in ds.paths]
for key in ['hydropathy', 'charge', 'rel_sasa']:
    c = [a[key] for a in attrs]
    plt.figure()
    plt.scatter(xy[:, 0], xy[:, 1], c=c, cmap='coolwarm', s=12)
    plt.colorbar(label=key)
    plt.title(f'latent PCA colored by {key}')
    plt.show()